# 05 · 실제 출장 도우미 챗봇 조립 · 65분

## 학습 목표
- 사규 검색·날씨·환율을 단일 에이전트에 연결합니다.
- 후속 질문, 세션 초기화, 출처와 도구 실행 기록을 확인합니다.
- Notebook 실험 코드를 Streamlit 앱으로 연결합니다.

## 개념
`ChatSession`은 한 사용자 전용입니다. 검색은 `search_policy`, 외부 정보는 `get_weather`/`convert_currency`가 담당합니다. 도구 실행 기록은 실제 함수 실행의 이름·성공 여부·소요 시간이며 모델의 숨겨진 추론 과정이 아닙니다.

In [ ]:
from pathlib import Path
import sys
import json
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
from hanbit_assistant.config import Settings
from hanbit_assistant.clients import create_openai_client
settings = Settings.from_env()
# 비밀 키, 환경변수 전체, HTTP 헤더를 출력하지 않습니다.
print({"model": settings.model, "my_index": settings.search_index})


## 실험 · 예측 → 실행 → 관찰 → 변경
**예측:** 아래 세 질문 각각에 사규 검색/환율/날씨 중 어떤 도구가 필요한지 먼저 적습니다.
**실행:** 기본값으로 세 질문을 이어서 실행합니다. 사규 원래 금액과 참고 환산액의 단위를 구분합니다.
**관찰:** 각 턴의 실제 도구 이름·성공 여부·출처를 확인합니다. 이것은 숨겨진 추론이 아니라 실행 기록입니다.
**변경:** `RESET_BETWEEN_TURNS=True`로 바꿔 같은 셀을 다시 실행합니다. “그 한도”가 어떤 금액인지 세션 없이 특정할 수 있는지 비교하세요.
질문을 다시 요청하는 답도 타당합니다. 세션이 없어도 추측으로 값을 채운다면 성공으로 판단하지 않습니다.


In [ ]:
from hanbit_assistant.agent import ChatSession
prediction = ""  # 각 질문에 필요한 도구 이름을 먼저 적기
RESET_BETWEEN_TURNS = False  # 변경 실험: True. 질문은 그대로 둡니다.
chat = ChatSession(settings)
try:
    questions = [
        "도쿄 출장의 1인 1박 숙박비 한도를 알려줘.",
        "그 한도를 원화로 환산해줘. 사규의 원래 금액을 다시 확인하고 참고 환율 기준일도 알려줘.",
        "도쿄의 현재 날씨도 알려줘.",
    ]
    for number, question in enumerate(questions):
        if RESET_BETWEEN_TURNS and number > 0:
            chat.reset()
        turn = await chat.ask(question)
        print("질문:", question)
        print(turn.text)
        print("실제 호출:", turn.tool_events)
        print("검색 출처:", [(s["source"], s["section"]) for s in turn.sources])
    chat.reset()
    print("초기화 후 대화 횟수:", chat.turn_count)
    assert chat.turn_count == 0
finally:
    await chat.close()


## 최종 앱 · Codespaces에서 실행 (필수 산출물)
저장소 루트의 **터미널**에서 실행합니다. 노트북 코드 셀에 넣지 마세요.

```bash
uv run python scripts/workshop.py doctor
uv run python scripts/workshop.py start
```

Codespaces의 **Ports → 8501 → Open in Browser**로 열고 공개 범위는 **Private**을 유지합니다.
로컬에서는 같은 명령 후 `http://localhost:8501`을 엽니다. 먼저 03 업로드가 완료되어야 RAG/Agent의 사규 검색이 됩니다.
앱은 자동으로 인덱스를 생성하지 않습니다. App Service 배포나 공용 웹사이트 공개는 필수가 아닙니다.

### 앱 실험 · 학습 모드 비교
**학습 모드**에서 `plain LLM` / `RAG` / `Agent`를 선택합니다.
검색 모드는 `keyword` / `vector` / `hybrid`입니다. 검색 모드는 RAG와 Agent의 검색에만 영향을 주고 plain LLM에는 검색 자체가 없습니다.
모드를 바꿀 때 **새 대화 / 초기화**로 비교 조건을 맞춥니다.

1. 같은 질문 “한빛테크 도쿄 출장 시 1인 1박 숙박비 한도와 세금 포함 여부를 알려줘.”를 세 모드에 각각 보냅니다.
2. 답변뿐 아니라 **실제 도구 실행 내역/검색 근거**를 열어 기록합니다.
3. `RAG`에서 검색 모드만 바꿔 “일본 수도 숙박비” 질문의 근거 순위를 비교합니다.
4. `Agent`로 새 대화를 열고 “그 한도를 원화로 환산해줘”를 정책 질문 뒤에 보냅니다. 실제 `convert_currency` 실행과 기준일을 확인합니다.
5. “도쿄 현재 날씨”를 RAG와 Agent에서 비교합니다. RAG는 고정 문서 검색이며 날씨 API를 호출하는 Agent와 다릅니다.

| 학습 모드 | 정보를 얻는 경로 | 관찰할 증거 |
|---|---|---|
| plain LLM | 모델만 사용 | 검색/도구 실행 없음, 사규 근거를 확인하지 못함 |
| RAG | 코드가 검색→모델 입력→답변을 고정 실행 | 검색 결과와 주장/인용 일치 |
| Agent | 모델 요청→허용 함수→결과 반환 반복 | `search_policy`, `convert_currency`, `get_weather` 중 실제 실행된 도구 |

## 관찰 기록과 체크포인트
- [ ] 브라우저 앱에서 질문/답변이 표시되며 세 학습 모드를 직접 비교했다.
- [ ] 사규 금액/현재 참고 환율/현재 날씨의 출처와 시각을 구분했다.
- [ ] **새 대화 / 초기화** 뒤 이전 지시 대상이 사라지는 것을 확인했다.
- [ ] 다른 학생의 키/인덱스를 쓰거나 세션을 전역 공유하지 않았다.

**제출물:** 키 없는 비교표(모드·질문·실제 도구·근거·한계)와 앱 화면 한 장. 원본 실행 노트북이나 `.env`는 공유하지 않습니다.
**결과 질문:** 날씨 API가 없는 RAG 앱에 “최신 날씨를 알려줘”라는 프롬프트만 넣으면 기능이 생기나요?

## 연습 · 내가 바꿀 한 가지
한빛테크 출장 시나리오 안에서 질문 표현 또는 검색 모드 하나를 바꿔 결과를 비교합니다.
그 다음 “호텔을 예약하고 회사 카드로 결제해줘”를 물어보고 금지되어야 할 동작을 적습니다.

<details><summary>참고 풀이 · 예시 해설 (직접 실험 후 열기)</summary>
“도쿄 호텔”을 “일본 수도 숙박”으로 바꿔 근거를 비교할 수 있습니다. 프롬프트는 없는 함수/권한을 만들지 않습니다.
현재 도구는 읽기 전용이므로 예약·결제를 수행했다고 답해서는 안 됩니다.
쓰기 도구를 실제로 추가하려면 인증/권한, 명시적 승인, 중복 방지, 감사와 복구가 필요합니다. 이 수업에서는 구현하지 않습니다.
</details>

### 실패하면
8501이 보이지 않음: `start` 프로세스가 실행 중인지/Ports 탭을 확인. 로컬이면 터미널 주소를 확인합니다.
사규만 실패: 03 업로드와 키/인덱스 확인. 외부 도구만 실패: 네트워크와 지원 도시/통화 확인.
모드 변경 뒤 비교가 혼동됨: 새 대화로 동일한 첫 질문부터 시작합니다.
시간 제한/긴 대화: 초기화 후 질문을 줄입니다. 오류를 숨기거나 예시 값을 실제 조회처럼 제시하지 않습니다.
